# DeepFakeBusted ? Colab Pro Teslim Ko?usu

Bu notebook mevcut projeyi bozmadan `xception_hf20k` adl? ikinci bir deney ko?usu ?retir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!cp "/content/drive/MyDrive/DeepFakeBustedDelivery/DeepFakeBusted_code.zip" .
!unzip -q DeepFakeBusted_code.zip -d /content
%cd /content/DeepFakeBusted


In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets


In [ ]:
!cp "/content/drive/MyDrive/DeepFakeBustedDelivery/current_dataset.zip" /content/
!unzip -q /content/current_dataset.zip -d /content/DeepFakeBusted/data/raw/archive/real_vs_fake/


In [ ]:
!python scripts/export_hf_binary_dataset.py \
  --dataset-id afatwapas/deepfake_face_classification \
  --dest-dir /content/hf_binary \
  --max-per-class 10000


In [ ]:
!python scripts/split_binary_dataset.py \
  --source-dir /content/hf_binary \
  --dest-dir /content/hf20k_split \
  --max-per-class 10000 \
  --clear-dest


In [ ]:
!mkdir -p results/checkpoints
!cp "/content/drive/MyDrive/DeepFakeBustedDelivery/checkpoints/xception_best.pth" results/checkpoints/


In [ ]:
!python -m training.train \
  --model xception \
  --extra-data-dir /content/hf20k_split \
  --run-name hf20k \
  --epochs 8


In [ ]:
!mkdir -p "/content/drive/MyDrive/DeepFakeBustedDelivery/new_results"
!cp results/checkpoints/xception_hf20k_best.pth "/content/drive/MyDrive/DeepFakeBustedDelivery/new_results/"
!cp results/logs/xception_hf20k_training.json "/content/drive/MyDrive/DeepFakeBustedDelivery/new_results/"


In [ ]:
!python -m training.evaluate --model xception
!python -m training.evaluate --model xception --run-name hf20k
!python -m training.evaluate --model xception --data-dir /content/hf20k_split
!python -m training.evaluate --model xception --run-name hf20k --data-dir /content/hf20k_split


## Hard-set de?erlendirmesi
`/content/hardset/real` ve `/content/hardset/fake` klas?rlerini olu?turarak kendi zor ?rneklerini buraya koy.


In [ ]:
!python scripts/evaluate_binary_folder.py \
  --model xception \
  --data-dir /content/hardset \
  --output-json /content/drive/MyDrive/DeepFakeBustedDelivery/new_results/xception_old_hardset.json

!python scripts/evaluate_binary_folder.py \
  --model xception \
  --run-name hf20k \
  --data-dir /content/hardset \
  --output-json /content/drive/MyDrive/DeepFakeBustedDelivery/new_results/xception_hf20k_hardset.json
